# Compare Performance of all Finetuned Models

## **DistillBERT**

- run_0
- run_1
- run_2

## **BioBERT**

- run_0

In [ ]:
import sys, os, json

PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

from gcp_utils import download_from_gcs, list_bucket_files
from config import settings

BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"
VERSION = settings.VERSION

# Load the labels (same for distill and biobert)
with open("data/distillbert_splits/test.jsonl", "r") as f:
    test_data = []
    for line in f:
        test_data.append(line)

with open("data/id2label.json", "r") as f:
    id2label = json.load(f)
with open("data/label2id.json", "r") as f:
    label2id = json.load(f)
# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}


# Helper to print summary
def run_summary(run_summary):
    # Unpack model information
    model_name = run_summary.get('model_name')
    training_time = run_summary.get('training_time_minutes')
    hyper = run_summary.get('hyperparameters', {})
    epoch = hyper.get('epoch')
    lr = hyper.get('lr')
    bs = hyper.get('batch_size')
    warmup_ratio = hyper.get('warmup_ratio')
    pth = hyper.get('push_to_hub')

    print("-"*20)
    print(f"\t\t🤖 Model: {model_name}")
    print("-"*20)
    print(f"⏱️ Training time: {training_time}")
    print("🛠️ Hyperparameters:")
    print(f"\t🚀 Epoch: {epoch}")
    print(f"\t🚀 LR: {lr}")
    print(f"\t🚀 Batch Size: {bs}")
    print(f"\t🚀 Warmup Ratio: {warmup_ratio}")
    print(f"\t⬆️ Push to hub: {pth}")
    return

def display_best_metrics(results_dict, model_name):
    import pandas as pd

    runs = []
    for run_name, metrics_dict in results_dict.items():
        row = {
            "run": run_name,
            "val_f1": metrics_dict["validation_metrics"]["f1"],
            "val_precision": metrics_dict["validation_metrics"]["precision"],
            "val_recall": metrics_dict["validation_metrics"]["recall"],
            "test_f1": metrics_dict["test_metrics"]["f1"],
            "test_precision": metrics_dict["test_metrics"]["precision"],
            "test_recall": metrics_dict["test_metrics"]["recall"],
        }
        runs.append(row)
    df = pd.DataFrame(runs).set_index("run")

    def get_max_info(column):
        idx_max = df[column].idxmax()
        val_max = df.loc[idx_max, column]
        return idx_max, val_max

    summary = []
    for col, label in [
        ("val_f1", "Validation F1"), 
        ("val_precision", "Validation Precision"), 
        ("val_recall", "Validation Recall"),
        ("test_f1", "Test F1"), 
        ("test_precision", "Test Precision"), 
        ("test_recall", "Test Recall")
    ]:
        run, val = get_max_info(col)
        summary.append((label, run, val))

    print(f"🏆 Best {model_name} runs by metric:\n")
    for label, run, val in summary:
        print(f"  • {label:<19s}: {run} ({val:.4f})")
    print("\nFull summary table:")
    display(df)

In [ ]:
VERSION

# DistillBERT Comparison

In [ ]:
# ==============================
# Load distillBERT models
# ==============================
MODEL_NAME = "distilbert-base-uncased" 
for idx in [0,1,2]:
    GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{idx}"
    LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{idx}"
    print(f"Downloading model from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )


In [ ]:
distill_val_test = {}
for idx in range(3):
    path = f"downloaded_models/distilbert-base-uncased/run_{idx}/summary.json"
    with open(path, "r") as f:
        summary = json.load(f)
    print(f"IDX: {idx}")
    run_summary(summary)
    distill_val_test[f"run_{idx}"] = {
        "validation_metrics" : summary["validation_metrics"],
        "test_metrics" : summary["test_metrics"]
    }

In [ ]:
display_best_metrics(distill_val_test, model_name = "DistilBERT")

*Best DistillBERT Model: run_0*

# BioBERT Comparison

In [ ]:
# ==============================
# Load BioBERT models
# ==============================
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"

for idx in [0,1,2]:
    
    LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{idx}"

    if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
        print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
        print("Skipping download from GCS.")
    else:
        GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{idx}"
        print(f"Downloading model from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
        downloaded_path = download_from_gcs(
            gcs_path=GCS_MODEL_PATH,
            local_path=LOCAL_MODEL_DIR,
            bucket_name=BUCKET_NAME
        )

In [ ]:
MODEL_NAME

In [ ]:
biobert_val_test = {}
for idx in range(3):
    path = f"downloaded_models/{MODEL_NAME}/run_{idx}/summary.json"
    if not os.path.exists(path):
        continue
    with open(path, "r") as f:
        summary = json.load(f)
    print(f"IDX: {idx}")
    run_summary(summary)
    biobert_val_test[f"run_{idx}"] = {
        "validation_metrics" : summary["validation_metrics"],
        "test_metrics" : summary["test_metrics"]
    }

In [ ]:
display_best_metrics(distill_val_test, model_name = "DistilBERT")

# BioBERT Top 2 Versus DistillBERT Top 1


In [ ]:
# BIOBERT 
# 1st place: run_2
# 2nd place: run_0

# DistillBERT 
# 1st place: run_0

# TEST SET COMPARISON

# Use os.path.join for proper path handling
bio_2 = os.path.join("downloaded_models", "dmis-lab", "biobert-base-cased-v1.1", "run_2", "test_metrics.json")
bio_0 = os.path.join("downloaded_models", "dmis-lab", "biobert-base-cased-v1.1", "run_0", "test_metrics.json")
distill_0 = os.path.join("downloaded_models", "distilbert-base-uncased", "run_0", "test_metrics.json")

paths = [
    distill_0,
    bio_0,
    bio_2
]

def print_test_metrics(metrics, model_name, run_id):

    print("---"*40)
    print(f"\t\tTEST RESULTS\n\tMODEL: {model_name} (run: {run_id}) ")
    print("---"*40)

    print("Token Level metrics: ")
    r = metrics['recall']
    a = metrics['accuracy']
    f1 = metrics['f1']
    p = metrics['precision']
    
    print(f"\tPrecision: {p:.4f}")
    print(f"\tRecall:    {r:.4f}")
    print(f"\tF1:        {f1:.4f}")
    print(f"\tAccuracy:  {a:.4f}")

    print("\nEntity Level metrics: ")
    
    # Extract all SYMPTOM_* entities
    entity_keys = [k for k in metrics.keys() if k.startswith("SYMPTOM_")]
    
    for entity_key in sorted(entity_keys):
        entity_metrics = metrics[entity_key]
        if isinstance(entity_metrics, dict):
            print(f"\t{entity_key}:")
            print(f"\t\tPrecision: {entity_metrics.get('precision', 0.0):.4f}")
            print(f"\t\tRecall:    {entity_metrics.get('recall', 0.0):.4f}")
            print(f"\t\tF1:        {entity_metrics.get('f1', 0.0):.4f}")
            print(f"\t\tNumber:    {entity_metrics.get('number', 'N/A')}")
    
    print("---"*40)
    print()


# Load metrics with error handling
for path_name, path, model_name, run_id in [
    ("distill_0", distill_0, "DistilBERT", "run_0"),
    ("bio_0", bio_0, "BioBERT", "run_0"),
    ("bio_2", bio_2, "BioBERT", "run_2")
]:
    if not os.path.exists(path):
        print(f"⚠️ Warning: File not found: {path}")
        print(f"   Current working directory: {os.getcwd()}")
        continue
    
    with open(path, "r") as f:
        metrics = json.load(f)
    print_test_metrics(metrics, model_name=model_name, run_id=run_id)


